# Phase 0: Fundamentals - Understanding Tree of Thoughts

## Overview

This notebook introduces the core concepts behind Tree of Thoughts without diving into code yet. We'll understand:
- Why standard prompting is limited
- How Tree of Thoughts solves those limitations
- Core concepts: Thoughts, Trees, Search, Evaluation
- When and why to use ToT

## Part 1: The Problem with Standard Prompting

### Chain-of-Thought (CoT) Baseline

Chain-of-Thought prompting improved reasoning by asking LLMs to "show their work":

```
Problem: If Sarah has 8 apples and buys 3 more, then eats 2, how many does she have?

CoT Response:
Step 1: Sarah starts with 8 apples
Step 2: She buys 3 more: 8 + 3 = 11
Step 3: She eats 2: 11 - 2 = 9
Answer: 9 apples
```

**CoT works well for many tasks**, but has limitations...

### Limitations of Linear Reasoning

#### 1. **Single Path Only**

CoT commits to one reasoning path. If the model takes a wrong turn, it continues down that wrong path:

```
Problem: Use numbers 3, 8, 3, 8 to make 24

Bad CoT Response:
Step 1: 3 + 8 = 11
Step 2: 11 + 3 = 14
Step 3: 14 + 8 = 22  ❌ Wrong! Can't recover

Better approach would be:
Step 1: 3 * 8 = 24  ✓ Found it!
```

The model might have found the right answer if it had explored alternative paths.

#### 2. **No Evaluation of Intermediate States**

CoT doesn't assess whether an intermediate step is "promising" or "likely to fail":

```
Problem: Write a novel chapter

CoT Response:
"The protagonist walked into the room...
He saw something strange...
It was a purple giraffe...
The giraffe spoke to him in Spanish..."

After 3 steps, the story is nonsensical, but CoT keeps going.
No mechanism to evaluate: "Is this getting better or worse?"
```

#### 3. **No Backtracking**

Once committed to a path, the model can't go back and try something different:

```
Problem: Find valid chess move

CoT Response:
Step 1: Move rook to A4
Step 2: This would expose the king... ❌ Illegal!
Step 3: Continue anyway with faulty logic

Ideal: Go back to Step 1 and try a different move
```

## Part 2: Enter Tree of Thoughts

### The Core Insight

**Problem-solving isn't always linear. It's navigating a tree of possibilities.**

When humans solve complex problems, we:
1. Consider multiple approaches
2. Evaluate which looks most promising
3. Explore that path deeper
4. If stuck, backtrack and try another path
5. Continue until we find the solution

Tree of Thoughts enables LLMs to do exactly this.

### Visual Comparison

#### Standard Prompting / Chain-of-Thought
```
START → Step1 → Step2 → Step3 → Step4 → END
        ↓       ↓       ↓       ↓
        ❌      ❌      ✓      ✓
        
If Step 2 is wrong, we're stuck.
Can't explore alternatives.
```

#### Tree of Thoughts
```
                    START
                   /  |  \
              Path1  Path2  Path3
              /        |      \
            P1a       P2a     P3a
           / | \      /       / \
          ✓  ❌ ✓    ❌      ❌  ✓
          |       |          |
        GOAL     GOAL       GOAL

Can explore multiple paths.
Can evaluate intermediate states.
Can backtrack if needed.
```

## Part 3: Core Concepts

### 1. **Thought**

A "thought" is an intermediate reasoning state.

**Examples:**

**Game of 24:**
```
Numbers: 3, 8, 3, 8

Thought 1: "Let me try multiplication: 3 * 8 = 24"
Thought 2: "Let me try addition chains: 3 + 8 = 11"
Thought 3: "Let me try subtraction: 8 - 3 = 5"
```

**Creative Writing:**
```
Task: Write opening paragraph

Thought 1: "Start with action scene"
Thought 2: "Start with character introduction"
Thought 3: "Start with atmospheric description"
```

**Key Property:** Each thought should be **evaluable** - the LLM can assess if it's a good direction.

### 2. **Tree Structure**

Thoughts are organized in a tree:

```
Root Thought: "Solve Game of 24 with [3, 8, 3, 8]"
       ↓
    /  |  \
   /   |   \          ← Level 1: Different first operations
3+8   3*8  8-3
 |      |    |
 ├──┐   |    └─→ 5         ← Level 2: Branches from each
 |  |   |       (result)
11  8  [next]
     ↓
    (Each continues...)
```

- **Node** = a thought (intermediate state)
- **Edge** = transition (generating next thought from current)
- **Leaf** = terminal state (final answer or dead end)
- **Path** = sequence of thoughts from root to leaf

### 3. **Evaluation / Value Function**

Not all thoughts are equally promising. We need to assess which branches are worth exploring.

**Two Evaluation Approaches:**

**Approach A: Value Function (Value-First)**
```
For thought "3 * 8 = 24":
  Ask LLM: "How likely is this to lead to the goal?"
  LLM: "This equals 24, which is our goal! Value: 1.0 (Excellent)"

For thought "3 + 8 = 11":
  Ask LLM: "How likely is this to lead to the goal?"
  LLM: "This is intermediate. We need to combine with 3,8 more. Value: 0.5 (Moderate)"
```

**Approach B: Vote Function (Vote-First)**
```
Ask LLM multiple times to vote on: "Is this a good path?"
  Vote 1: "Yes, promising"
  Vote 2: "Not very promising"
  Vote 3: "Yes, promising"
  Result: 2/3 votes = moderate confidence
```

**Purpose:** Use evaluation to prune low-value branches and focus on promising ones.

### 4. **Search Strategy**

How do we navigate the tree to find the solution efficiently?

#### **Depth-First Search (DFS)**
```
Strategy: Go deep on one branch before backtracking
Order: 1 → 2 → 3 → 4 → back to 1 → 5 → 6

        1
       / \
      2   5
     / \   \
    3   4   6

Pros: Memory efficient, finds deep solutions
Cons: May waste time on dead-end branches
```

#### **Breadth-First Search (BFS)**
```
Strategy: Explore all options at each level
Order: 1 → 2 → 5 → 3 → 4 → 6

        1
       / \
      2   5        Level 1: Check both
     / \   \       Level 2: Check all children
    3   4   6

Pros: Finds shallowest solution
Cons: Explores many branches
```

#### **Best-First / Beam Search**
```
Strategy: Always explore most promising branch
Order: 1 → 5 (highest value) → 6 (highest from 5)

        1 ← Start here
       / \  Value scores: 2=0.4, 5=0.9
      2   5 ← Go here (highest value)
     / \   \
    3  4   6 ← Only explore top-k branches

Pros: Most efficient, focuses on promising paths
Cons: May miss better solutions in lower-value branches
```

## Part 4: The ToT Algorithm (High Level)

### Pseudocode

```
FUNCTION TreeOfThoughts(problem):
    
    1. root_thought ← Initial problem statement
    2. frontier ← [root_thought]  // Queue of nodes to explore
    3. best_solution ← None
    
    WHILE frontier not empty AND haven't found solution:
        
        4. current_thought ← pop from frontier (based on strategy)
        
        // Check if we reached goal
        5. IF is_solution(current_thought):
               best_solution ← current_thought
               RETURN best_solution
        
        // Generate candidate next thoughts
        6. candidates ← call_llm(generate_prompt(current_thought))
        
        // Evaluate each candidate
        7. FOR EACH candidate IN candidates:
               value ← call_llm(evaluate_prompt(candidate))
               candidate.value ← value
        
        // Add promising candidates to frontier
        8. promising ← filter candidates by value threshold
        9. frontier.add(promising)
        
        // Optional: Prune low-value branches
        10. IF frontier too large:
                frontier ← keep only top-k by value
    
    11. RETURN best_solution
```

### Key Steps

1. **Initialize**: Start with problem statement
2. **Generate**: Ask LLM for candidate next thoughts
3. **Evaluate**: Ask LLM to score each candidate
4. **Search**: Use search strategy to navigate tree
5. **Repeat**: Continue until solution found or tree exhausted

## Part 5: When to Use Tree of Thoughts

### ✅ ToT Works Well When:

1. **Multiple solution paths exist**
   - Math problems (multiple ways to reach 24)
   - Planning problems (different sequences)
   - Creative tasks (different valid approaches)

2. **Intermediate evaluation is possible**
   - Clear progress indicators
   - Can assess "Am I getting closer?"
   - Partial solutions can be verified

3. **Backtracking is valuable**
   - Dead-end paths are common
   - Recovering from mistakes helps
   - Exploration pays off

4. **Problems are moderately complex**
   - Can't be solved in single step
   - But solvable in reasonable depth (3-5 steps)
   - Requires reasoning beyond simple pattern matching

### ❌ ToT Less Suitable When:

1. **Single obvious path**
   - Step-by-step linear process
   - No branching or decision points
   - CoT is sufficient

2. **No intermediate evaluation possible**
   - Only final answer can be checked
   - Can't assess intermediate progress
   - All paths equally opaque

3. **Simple factual tasks**
   - Retrieval of knowledge
   - Single answer lookup
   - No reasoning needed

4. **Cost is critical**
   - ToT can be 5-20x more expensive
   - Multiple LLM calls needed
   - Budget constraints prohibit it

## Part 6: Real Example from Paper

### Game of 24

**Problem:** Given four numbers [3, 8, 3, 8], use arithmetic to make 24

#### Standard Prompting Fails

```
Problem: Make 24 from [3, 8, 3, 8]

Response: 
Let me try: 3 + 8 + 3 + 8 = 22. Not quite.
Let me try: 3 * 8 + 3 + 8 = 35. Too much.
I can't figure it out.

Result: ❌ Fails
```

#### Chain-of-Thought Sometimes Works

```
Problem: Make 24 from [3, 8, 3, 8]

Response:
Step 1: 8 / (3 - 8/3) = 8 / (1/3) = 24 ✓

Result: ✅ Works (but requires finding right path)
```

#### Tree of Thoughts Systematically Explores

```
Root: [3, 8, 3, 8]

Level 1 (Generate 5 candidates):
  - Path A: 3 * 8 = 24 (Value: 1.0 - FOUND IT!)
  - Path B: 3 + 8 = 11 (Value: 0.3 - needs work)
  - Path C: 8 - 3 = 5 (Value: 0.2 - uncertain)
  - Path D: 3 / 8 = 0.375 (Value: 0.1 - seems bad)
  - Path E: (8-3)*8 = 40 (Value: 0.4 - too large)

Result: ✅ Found solution efficiently!
  Also tried other paths for comparison
```

**Paper Results on Game of 24:**
- Standard Prompting: 4% success rate
- Chain-of-Thought: 30% success rate  
- Tree of Thoughts: 74% success rate (4-step DFS with value heuristic)

**That's 2.5x improvement over CoT!**

## Part 7: Key Takeaways

### The Innovation

Tree of Thoughts shifts problem-solving from **linear to exploratory**:

| Aspect | CoT | ToT |
|--------|-----|-----|
| **Paths** | Single linear | Multiple branching |
| **Evaluation** | None (just generate) | Intermediate step scoring |
| **Backtracking** | Not possible | Full backtracking support |
| **Search** | Implicit (sequential) | Explicit (DFS/BFS/Best-First) |
| **Complexity** | Simple | More complex |
| **Cost** | Low | Medium-High |
| **Accuracy** | Good for simple tasks | Excellent for complex tasks |

### When It Matters Most

ToT provides biggest improvements on tasks where:
1. Multiple solution paths exist
2. Intermediate steps can be evaluated
3. Backtracking/recovery is valuable
4. Cost-accuracy trade-off is acceptable

### The Cost-Accuracy Trade-Off

```
Accuracy
   |
   |                    ToT (Beam Search)
 95|                   /
   |                  /
   |              ToT (DFS)
 75|              /
   |             /
   |        CoT /
 50|        /
   |       /
   |   Standard/
 10|________/_____________________ Cost (API Calls)
   0       10      100      1000+

Deeper trees and wider beams = more expensive but more accurate
```

## Part 8: Next Steps

Now that you understand the **concepts**, you're ready to:

### Phase 1: Basic Concepts (Practical)
- Implement thought trees in code
- Understand data structures
- Work with simple manual examples

### Phase 2: Algorithms (Implementation)
- Code up DFS, BFS, Best-First search
- Build evaluation functions
- Optimize tree exploration

### Phase 3: LLM Integration (Integration)
- Prompting for thought generation
- Prompting for state evaluation
- Connecting to real LLM APIs

### Phase 4: Complete System (Advanced)
- Build end-to-end ToT system
- Solve real problems
- Optimize and deploy

---

**Ready? Open `phase1_basic_concepts.ipynb` to start building!**